# DQN CartPole Demo

This notebook is a thin debugging surface for the repo-native DQN workflow. The implementation lives in `model/rl_workflow.py`; this notebook only calls those functions.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.rl_workflow import (
    evaluate_dqn,
    load_dqn_checkpoint,
    make_dqn_config,
    train_dqn,
)

print(ROOT)

## Smoke Run

Use the smoke profile for a quick end-to-end check. Switch to `fast` for the CartPole solve target.

In [ ]:
cfg = make_dqn_config("smoke", seed=0)
state = train_dqn("cartpole", cfg, progress=True)
state["summary"]

## Plot Episode Returns

In [ ]:
import csv
import matplotlib.pyplot as plt

run_dir = Path(state["run_dir"])
rows = []
with (run_dir / "training_metrics.csv").open(newline="", encoding="utf-8") as handle:
    for row in csv.DictReader(handle):
        rows.append(row)

steps = [int(row["global_step"]) for row in rows]
returns = [float(row["episode_return"]) for row in rows]

plt.figure(figsize=(8, 4))
plt.plot(steps, returns)
plt.xlabel("Global step")
plt.ylabel("Episode return")
plt.title("DQN CartPole training returns")
plt.grid(alpha=0.3)
plt.show()

## Reload And Evaluate

In [ ]:
loaded = load_dqn_checkpoint(state["checkpoint_path"])
metrics = evaluate_dqn(
    loaded["task"]["name"],
    loaded["model"],
    episodes=cfg.eval_episodes,
    seed=cfg.eval_seed,
    device=loaded["device"],
)
{k: v for k, v in metrics.items() if k != "episode_metrics"}